# Overview

- This is a very dense course that many conceptual ideas
- Thus, it is very useful to have a document upfront that provides an overall view of what we are trying to build
- This will tie all the components together, before we do a deep dive into individual components

- At a very high level, this is how software gets translated to hardware:
    1. Software Layer: Human writes something in Python/C++/Scala etc
    2. Compiler: Translates code into binary machine instructions
    3. Instruction: Binary numbers act as physical keys
    4. Control Unit (ALU): Routes control signals to specific circuits
    5. Transistors: PMOS/NMOS switches open/close

- We will build up our understanding from the bottom most layer upwards

## Transistors --> Logic Gates (Nand, And, Or)

- The purpose of this section is to take us from Transistors --> Nand gates
    - Nand gates are universal building blocks in computing
    - We will not go into the physics of creating transistors, or how they are doped
    - As far as we are concerned, we treat transistors as the most atomic building block
    - From this basic hardware atom, springs the entire universe of computing

- So what exactly is a transistor?
    - Think of transistors as switches that responds in a fixed way to voltage
    - You can think of these as basic mechanical switches with 2 inputs; (i) One input comes from the main power source, which is always 1 when the power is on, and (ii) the second input is what drives behaviour, and it is supplied by the output wire of another transistor, an external input pin (like a keyboard switch), or the system clock.

- A transistor can output 2 states:
    - Either it passes through the power that is being supplied, which reads as 1
    - Or it passes GROUND, which reads as None
    - Let's fix the 2 states here as global constants

In [ ]:
POWER = True
GROUND = None
type TRANSISTOR_OUTPUT = POWER | GROUND

### Basic Transistors: NMOS + PMOS

In [ ]:
- There are 2 basic types of transistors we need to know about, that form the building blocks for basic logical gates
    - A PMOS transistor 
        - P-type Metal-Oxide-Semiconductor
        - Also known as a "Pull-Up Team"
        
    - A NMOS transistor
        - N-type Metal-Oxide-Semiconductor
        - Also known as a "Pull-Down Team"

In [ ]:
def pmos(gate_signal: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    """
    Pass through source signal when gate_signal is False (i.e. low power)
    None when gate_signal is True (i.e. high power)
    """
    return source if not gate_signal else None

def nmos(gate_signal: TRANSISTOR_OUTPUT, source: bool = POWER) -> POWER | GROUND:
    """
    Pass through source signal when gate_signal is True (i.e. high power)
    None when gate_signal is False (i.e low power)
    """
    return source if gate_signal else None

In [ ]:
print(
    pmos(True), pmos(False),
    nmos(True), nmos(False)
)

### Transistors to Logic

- From these 2 signals, we can start building out logic gates!

- Fun fact; in electronics, NAND and NOR are much simpler to build than NOT/AND/OR 
    - So we will start by building out NAND and NOR using both PMOS and NMOS transistors

- In the code below, we replicate the effect of the transistor -> logic gates via code. 
    - One source of confusion: you might notice we use logical expressions `and`, `or` to build out these logic gates
    - It probably seems circular that we are creating logic gates from logic expressions, but keep in mind that this is just a representation of how the transistor works. 
    - In the physical world, we rely on physical arrangement of the circuit (parallel, series) to derive this effect

- Every function below comes with an ASCII sketch of the circuit to show what I mean

In [ ]:
def pmos_NAND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                     POWER (1)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [PMOS A]    [PMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        ├─── OUTPUT (Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''
    # If input1 and input2 are both True, both PMOS return None, and so no output is returned
    # Otherwise, so long as one of input1/input2 are False, the power flows through one of the PMOS paths and 
    # output returns True
    pmos_A = pmos(input1, source)
    pmos_B = pmos(input2, source)
    
    return POWER if pmos_A or pmos_B else GROUND

def pmos_NOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [PMOS A] <-- Input1
            │             
        [PMOS B] <-- Input2
            │
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)
    '''
    # If both input1 and input2 are None, both PMOS transistors let power through, and Output is POWER
    # Else if either input1 or input2 are True, the circuit breaks, and ground is returned
    pmosA = pmos(input1, source)
    pmosB = pmos(input2, pmosA)

    return POWER if pmos_B else GROUND

def pmos_NOT(input1: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
         [PMOS] <-- Input1
            │
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)
    '''
    ## PMOS negates the input by construction, because it outputs 0 if input is 1
    out1 = pmos(input1, source)
    return POWER if out1 else GROUND

def pmos_AND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                     POWER (1)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [PMOS A]    [PMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        ├─── NAND Output
                        │
                    [Resistor]
                        │
                    GROUND (0)
                        │
                     POWER (1)
                        │
                    [PMOS NOT] <-- NAND Output
                        │
                        ├─── OUTPUT (AND Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''

    ## - If both input1 and input2 are 1, then the NAND Output is 0
    ##    - If the NAND output is 0, the NOT output is 1
    ## - Else the NAND output is 1
    ##    - If the NAND output is 1, the NOT output is 0
    ## - Therefore, the truth table becomes equivalent to AND
    ##    - (1,1) -> 1
    ##    - (0,1) -> 0
    ##    - (1,0) -> 0
    ##    - (0,0) -> 0

    nand_out = pmos_NAND(input1, input2, source=source)
    return pmos_NOT(nand_out, source=source)

def pmos_OR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                    POWER (1)
                        │
     Input1 -------> [PMOS A]
                        │             
     Input2 -------> [PMOS B]
                        │
                        ├─── Intermediate (NOR)
                        │
                    [Resistor]
                        │
                    GROUND (0)
                        │
                     POWER (1)
                        │
                    [PMOS NOT] <-- Intermediate (NOR)
                        │
                        ├─── OUTPUT (OR Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''
    ## - If both input1 and input2 are 0, then the NOR Output is 1
    ##    - If the NOR output is 1, the NOT output is 0
    ## - Else the NOR output is 0
    ##    - If the NOR output is 0, the NOT output is 1
    ## - Therefore, the truth table becomes equivalent to OR
    ##    - (1,1) -> 1
    ##    - (0,1) -> 1
    ##    - (1,0) -> 1
    ##    - (0,0) -> 0
    nor_out = pmos_NOR(input1, input2, source=source)
    return pmos_NOT(nor_out, source=source)

In [ ]:
def pmos_NAND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    ...

def pmos_NOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    ...

def pmos_NOT(input1: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    ...

def pmos_AND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    ...

def pmos_OR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    ...

### CMOS

- Using PMOS and NMOS, we can now build a CMOS transistor
    - Complementary Metal-Oxide-Semiconductor
    - "Complementary" because PMOS and NMOS work as a coordinated pair:
        - PMOS (Pull-Up): Connects the output to Power (1 / True) when activated by 0
        - NMOS (Pull-Down): Connects the output to Ground (0 / False) when activated by 1
    - This dual design ensures output is always driven cleanly to high or low, preventing short circuits or "floating" states.

In [ ]:
- CMOS NAND

                POWER (1)
               ┌─────┴─────┐
               │           │
           [PMOS A]    [PMOS B]  <-- Parallel Pull-Up Network
               │           │
               └─────┬─────┘
                     ├─── OUTPUT (Out)
               ┌─────┴─────┐
               │           │
           [NMOS A]        │
               │           │     <-- Series Pull-Down Network
           [NMOS B]        │
               │           │
             GROUND (0) ───┘

- PMOS NAND

      POWER (1)
    ┌─────┴─────┐
    │           │
[PMOS A]    [PMOS B]   (Parallel Network)
    │           │
    └─────┬─────┘
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)

- NMOS NAND

